# మనిషి-ఇన్-ది-లూప్: ప్రీ-యాక్షన్ గేట్స్, రిస్క్ టియర్, మరియు ఆడిట్ లాగింగ్

ఈ పాఠానికి README ఒక చిన్న స్నిపెట్‌తో మనిషి-ఇన్-ది-లూప్ ను పరిచయం చేస్తుంది, ఇందులో ఏజెంట్ ఇప్పటికే ఒక ప్రతిస్పందనను ఉత్పత్తి చేసిన తరువాత వాడుకరి `APPROVE` లేదా `REJECT` అడుగుతుంది. ఆ నమూనా మంచి ప్రారంభబిందువు అయినప్పటికీ, ఉత్పత్తి HITL అమలుకోలకు సాధారణంగా మూడూ అదనపు భాగాలు అవసరం ఉంటాయి:

1. ఏజెంట్ ఒక ప్రమాదకరమైన దశను నిర్వహించే ముందు నడిచే **ప్రీ-యాక్షన్ గేట్**, తద్వారా ఖర్చు, తిరుగుబాటు అవకాశంలేమీ మరియు ఆలస్యం నియంత్రణలో ఉంటాయి.
2. **రిస్క్ టియర్**, అందువల్ల తక్కువ-రిస్క్ చర్యలు ఆటోమేటిక్‌గా అమలవుతాయి, మధ్య-రిస్క్ చర్యలు బ్యాచ్ ఆమోదం పొందుతాయి, మరియు కేవలం ఉన్నత-రిస్క్ చర్యలు మాత్రమే మనిషిపై బ్లాక్ అవుతాయి.
3. ఒక **ఆడిట్ లాగ్ మరియు సవరణ లూప్**, అందువల్ల ప్రతి గేట్ నిర్ణయం JSONL గా రికార్డ్ అవుతుంది, మరియు తిరస్కరణ కారణంతో structured రీప్రమ్ప్ట్ చేసి ఏజెంటుకి పంపబడుతుంది, కేవలం `Revising...` ను ప్రింట్ చేయకుండా.

ఈ నోట్‌బుక్ `06-system-message-framework.ipynb` లోని అదే ప్రిమిటివ్స్ పై ప్రతి ఒకటి నిర్మిస్తుంది. ఇది `DEMO_MODE = True` లో (ఇంటరాక్టివ్ ఇన్‌పుట్ అవసరం లేదు) లేదా `DEMO_MODE = False` నిర్ధారించుకున్నప్పుడు واقعی `input()` ప్రాంప్ట్‌లతో ఎండ్-టూ-ఎండ్ నడుస్తుంది. గమనిక: DEMO_MODE లో మూడవ లక్ష్యం అయిన రీట్రై స్క్రిప్టెడ్ ఉంది కాబట్టి లూప్ మెకానిక్స్ పూర్తి వివరంగా దర్శనమవుతుంది. నిజమైన సవరణ ఆధారిత పునఃవర్గీకరణకు `DEMO_MODE = False` మరియు ఆపరేటర్ అవసరం.

**వ్యాప్తికి లోపం (ఇతర పాఠాల్లో నిర్వహించబడుతుంది):** ప్రమాణపత్రీకరణ మరియు యాక్సెస్ నియంత్రణ (పాఠం 06 README ముప్పు 2), టూల్-కాల్ మధ్యవర్తితనం (పాఠం 14 MAF లో లోతైన అధ్యయనం), బహుళ-ఏజెంట్ వాదన నమూనాలు.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## నమూనా 1: ప్రీ-యాక్షన్ గేట్

READMEలో HITL స్నిపెట్ మొదటే ఏజెంట్‌ను పిలుస్తుంది, తరువాత యూజర్‌ను అవుట్‌పుట్‌ను ఆమోదించమని అడుగుతుంది. అది **పోస్ట్-యాక్షన్** ఫ్లో. ఏజెంట్ ఇప్పటికే అమలు చేశాడు, అందువలన LLM కాల్ ఖర్చు ఇప్పటికే చెల్లించబడింది, మరియు ఏదైనా సైడ్ ఎఫెక్ట్ (ఇమెయిల్ పంపడం, డేటాబేస్ రో వ్రాయడం, కామెంట్ పోస్ట్ చేయడం) ఇప్పటికే జరిగింది.

**ప్రీ-యాక్షన్** ఫ్లో ఏజెంట్ ప్రమాదకరమైన దశను మొదలుపెట్టే ముందు గేట్‌ను చేర్చుతుంది. ఏజెంట్ చర్యను ప్రతిపాదిస్తాడు, గేట్ అమలు చేయాలా అనేదాన్ని నిర్ణయిస్తుంది, మరియు ఆమోదం ఉన్నపుడే సైడ్ ఎఫెక్ట్ జరుగుతుంది.

| అంశం | పోస్ట్-యాక్షన్ ఆమోదం (README స్నిపెట్) | ప్రీ-యాక్షన్ గేట్ (ఈ నోట్‌బుక్) |
|---|---|---|
| ఆమోదం ఎప్పుడు జరుగుతుంది? | ఏజెంట్ అవుట్‌పుట్ రూపొందించిన తర్వాత | ఏదైనా సైడ్-ఎఫెక్ట్ అమలు చేసే ముందు |
| తిరస్కరణపై LLM ఖర్చు | ఇప్పటికే చెల్లించబడింది | ప్రతిపాదనకు మాత్రమే చెల్లించబడింది, చర్యకు కాదు |
| తిరిగి తిప్పలేని సైడ్ ఎఫెక్ట్స్ | సంభవించవచ్చు (చర్య ఇప్పటికే జరిగింది) | నివారించబడింది |
| ఆడిట్ స్పష్టత | ఆమోదం ఒక ప్రింట్ స్టేట్‌మెంట్ | ఆమోదం టైమ్స్‌టాంప్, చర్య, కారణం తో ఒక JSON రికార్డు |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## నమూనా 2: రిస్క్ స్థాయి phân చేర్పు

ప్రతి చర్యకు మానవ నిమిత్తం ఆమోదం అవసరం లేదు. ఒక పబ్లిక్ API పై రీడ్-ఒన్లీ లుకప్ ఒక కస్టమర్ ఇమెయిల్ పంపించడంపై భిన్నంగా ఉంటుంది. రెండింటినీ సమానంగా చూడటం ఆపరేటర్ దృష్టిని వృథా చేస్తుంది మరియు ఏజెంట్ నెమ్మదిగా చేస్తుంది.

ఒక సరళమైన 3-స్థాయి మోడల్:

| స్థాయి | ఉదాహరణలు | ఆమోద ప్రవాహం |
|---|---|---|
| `తగ్గిన` (రీడ్-ఒన్లీ) | జ్ఞానాధారాన్ని శోధించడం, ఫ్లైట్ ఎంపికలను చూడటం, ఒక పబ్లిక్ వెబ్ పేజీని తీసుకోవడం | ఆటో-నిర్వహణ, ఆడిట్ కొరకు లాగ్ చేయబడింది |
| `మధ్యమ` (సులభమైన మార్పు) | ఒక ఫలితాన్ని క్యాశ్ చేయడం, కౌంటర్‌ను పెంచడం, ఒక రిమైండర్‌ను షెడ్యూల్ చేయడం | ఆటో-నిర్వహణ, కానీ రోజువారీ సమీక్షలో కలిపి ఉంటుంది |
| `ఎత్తైన` (బాహ్య-ముఖ్యమైన లేదా తిరగలేని) | ఇమెయిల్ పంపించడం, కార్డును ఛార్జ్ చేయడం, ఒక పబ్లిక్ ఛానెల్‌లో పోస్ట్ చేయడం | మానవ ఆమోదం పై బ్లాక్ చేయండి |

ఇది ఒక స్థాయి phân చేర్పు. ఉత్పత్తి వ్యవస్థలు తరచుగా మరింత నాజూకు స్థాయిలను ఉపయోగిస్తాయి (ఉదా: AWS IAM అనుమతి స్థాయిలు, పాత్ర ఆధారిత యాక్సెస్ స్థాయిలు). కింది 3-స్థాయి సంస్కరణ చదవ-only మరియు సైడ్-ఎఫెక్టింగ్ చర్యలను కలిపిన ఏజెంట్ కోసం చాలా చిన్న ఉపయోగకరమైన సంస్కరణ.

దిగువ క్లాసిఫയർ కీవర్డ్ heuristics ఉపయోగిస్తుంది కాబట్టి డెమో తేలికగా మరియు నిర్ణీతంగా ఉంటుంది. ఉత్పత్తి వ్యవస్థలో మీరు దీన్ని నేర్చుకున్న క్లాసిఫയർ లేదా పాలసీ ఇంజిన్ తో మార్చేస్తారు.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## నమూనా 3: ఆడిట్ లాగ్ మరియు సవరణ లూప్

ఒక `print("Response approved.")` ఆడిట్ లాగ్ కాదు. నమ్మకాన్ని కొరకు, ప్రతి గేట్ నిర్ణయాన్ని ఒక నిర్మితమైన ఘటనగా జోడించాలి, దీన్ని మీరు తర్వాత ప్రశ్నించవచ్చు, పునరావృతం చేయవచ్చు లేదా సంఘటన సమీక్షకు జోడించవచ్చు.

రెండు భాగాలు:

1. **అప్పెండ్-ఆన్లీ JSONL.** ఒక్కో నిర్ణయానికి ఒక పঙక్తి, టైంశీట్తో, చర్య, స్థాయి, నిర్ణయం, కారణం. grep చేయడానికి సులభం, తర్వాత నిజమైన లాగ్ స్టోర్‌కు పంపించడమూ సులభం.
2. **అంగీకారం లేనప్పుడు సవరణ లూప్.** గేట్ `deny` తిరిగి ఇచ్చినప్పుడు, ఏజెంట్ తనను తానే తిరిగి ప్రశ్నిస్తుంది తిరస్కరణ కారణంతో సారాంశంలో, తద్వారా తరువాతి ప్రతిపాదన సమస్యను నివారించగలదు.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## అదనపు వనరులు

ఈ HITL నమూనాలను వివిధ రకాలుగా అమలు చేసే మరికొన్ని ఇతర పబ్లిక్ ప్రాజెక్టులు ఉన్నాయి. మీ స్టాక్‌కు సరిపోయేవి కనుగొనడానికి వాటి దృక్కోణాలను పోల్చండి:

- **LangChain** మానవ-ఇన్-ది-లూప్ టూల్ రాప్పర్స్ ([డాక్స్](https://python.langchain.com/docs/integrations/tools/human_tools)): మానవ ఇన్‌పుట్ కోసం ఎగ్జిక్యూషన్‌ను ఆపివేసే డ్రాప్-ఇన్ టూల్ రాప్పర్స్.
- **AutoGen** `UserProxyAgent` ([v0.2 డాక్స్](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ దీనిని మళ్లీ నిర్మించింది): బహుళ-ఏజెంట్ సంభాషణల్లో మానవుని ప్రాతినిధ్యం చేసే ఏజెంట్ పాత్రను ఉపయోగిస్తుంది.
- **Microsoft Agent Framework (MAF)** ఫంక్షన్-ఇన్వొకేషన్ మిడిల్‌వేర్ ([డాక్స్](https://learn.microsoft.com/agent-framework/)): ప్రతి టూల్/ఫంక్షన్ కాల్ చుట్టూ నడిచే మిడిల్‌వేర్, ఇంతకు తగిన గేటింగ్ లాజిక్ మరియు ఆమోద ఫ్లోలకు తగినది.

ప్రతి ప్రాజెక్ట్ ఈ మూడు ఉప-నమూనాలను వేరుగా నిర్వహిస్తుంది: LangChain వాటిని టూల్స్‌గా రాప్పర్ చేస్తుంది, AutoGen ఏజెంట్ పాత్రను ఉపయోగిస్తుంది, Microsoft Agent Framework ఫంక్షన్-ఇన్వొకేషన్ మిడిల్‌వేర్‌ను ఉపయోగిస్తుంది. మీ స్వంత ఏజెంట్‌కు డిజైన్‌ను ఎంచుకునే ముందు ఒకటి లేదా రెండు అమలు విధానాలను పూర్తిగా చదవండి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
